# Benchmark III: a 123-qubit angle family that loads the real table exactly

**What this notebook establishes, and what it does not.**

It builds `ANGLE_k` circuits for the dfthcsos submission at $brot = 10$ bits per word and
$J = 10$ words per circuit, and checks them with the competition's own
`scoreinfra.circuits`: the register contract, word recovery on all 3,976 addresses, the
phase, the workspace, and the reverse pass. It fits the 128-qubit cap with **five qubits
spare**, uses **eight** circuits for 75 angles, and reproduces all 298,200 angle words
bit-exactly.

§6 then scores that family, and every other precision the cap admits, with
`dfthcsos/verify.py` against the `FeMoCo76o113e.fcidump` in the competition mount. The
cheapest family is **inadmissible**: `error_1` is 242 mHa against a 20 mHa limit, so the two
bits of precision §2 spends are not there to spend, and the score belongs to the 12-bit
family the bundle already ships. The verification of the circuits stands; the trade does
not.

Three findings, in the order they arrived:

1. **Toffoli cost is set by the address tree, not by word width.** Measured 3,974
   Toffoli-like gates per circuit against a 3,975-node full tree, at every `(brot, J)`
   tried. Precision therefore costs almost nothing in `T` and a lot in qubits.
2. **That makes the batch the lever, and the accuracy gate the constraint.** Dropping from
   12 to 10 bits buys $J = 10$ instead of $J = 8$, which cuts the family from 10 circuits
   to 8 and its two-pass Toffoli count from 79,480 to **63,584** — 19.98% for 4 extra
   qubits. §6 measures what that costs at the gate: 242 mHa of one-body residual, so the
   19.98% is not purchasable at this precision.
3. **The 9-bit delta-compression idea is false on this table.** The Givens sign invariant
   `bit 9 == bit 8` holds on only **78.59%** of words, because this table's angles span
   $\theta/\pi \in [0, 2)$ rather than the $\pm\pi/2$ the argument assumed. Recorded as a
   falsification, with the measurement that kills it.


## 1. Locate the competition bundle

The circuits are checked by the bundle's own code, so the bundle must be mounted. A run
that silently used a local stand-in would be reporting on a different verifier, so this
cell stops if it cannot find one.

In [ ]:
import sys, pathlib

ROOTS = [pathlib.Path("/kaggle/input"), pathlib.Path.cwd(), pathlib.Path.cwd() / ".."]
bundle = None
for root in ROOTS:
    for cand in sorted(root.glob("**/verifiers/dfthcsos")) if root.is_dir() else []:
        bundle = cand.parent.parent
        break
    if bundle:
        break
assert bundle is not None, "competition bundle not found; mount the dfthcsos data first"

sys.path.insert(0, str(bundle / "verifiers"))
sys.path.insert(0, str(bundle / "kickmix"))
import numpy as np
import kickmix as km
from dfthcsos.scoreinfra import circuits as C

npz = next(p for p in sorted((bundle / "sample_submission" / "dfthcsos").glob("*.npz")))
with np.load(npz, allow_pickle=False) as z:
    words12 = np.array(z["angle_words"], dtype=np.int64)
print("bundle:", bundle.name, "| kickmix has mr:", hasattr(km.CircuitBuilder, "mr"))
print("angle_words:", words12.shape, "range", words12.min(), words12.max())

## 2. The cap is a product, not a width

Each `ANGLE_k` circuit must satisfy `num_qubits <= 128`. Its width is

$$Q = \underbrace{12}_{\text{selection: } \lceil\log_2 3976\rceil}
    + \underbrace{J \cdot brot}_{\text{word registers}}
    + \underbrace{11}_{\text{tree ancillas } = 12-1}$$

The selection width is fixed by the row count and the contract rejects extra selection
bits; the ancillas are fixed by the tree. Only $J \cdot brot$ is free, so the cap binds the
**product** — and because the number of circuits is $K = \lceil 75/J \rceil$, the cheapest
family is the largest $J$ that still fits.

In [ ]:
def requantise(words, src_bits, dst_bits):
    """Move a word table to dst_bits, keeping theta/pi = word / 2**(b-1).

    Words are turn counts modulo one turn, so a rounded 4095 becomes 1024 and must wrap
    to 0; without the mask the caller gets a word one past the register width.
    """
    words = np.asarray(words, dtype=np.int64)
    mask = (1 << dst_bits) - 1
    if dst_bits >= src_bits:
        return (words << (dst_bits - src_bits)) & mask
    shift = src_bits - dst_bits
    return ((words + (1 << (shift - 1))) >> shift) & mask


def batch_deltas(words, batch_size):
    """delta[0] = batch[0]; delta[k] = batch[k] XOR batch[k-1]. Shape (K, A, J)."""
    rows, angles = words.shape
    k = max(1, -(-angles // batch_size))
    padded = np.zeros((rows, k * batch_size), dtype=np.int64)
    padded[:, :angles] = words
    batches = padded.reshape(rows, k, batch_size).transpose(1, 0, 2)
    deltas = batches.copy()
    deltas[1:] ^= batches[:-1]
    return deltas


def build_delta_circuit(delta, brot):
    """One XOR oracle: a segment-tree QROM over the selection register."""
    rows, J = delta.shape
    nbits = (rows - 1).bit_length()
    b = km.CircuitBuilder()
    sel = b.create_quantum_register(nbits, C.SELECTION_REGISTER)
    words = [b.create_quantum_register(brot, C.word_register(j)) for j in range(J)]
    anc = b.alloc_qubits(nbits - 1) if nbits > 1 else []
    bits = [sel[i] for i in reversed(range(nbits))]

    def load(row, ctl):
        for j in range(J):
            w = int(delta[row, j])
            for bit in reversed(range(brot)):
                if (w >> bit) & 1:
                    b.cx(ctl, words[j][bit])

    def node(ctl, level, lo, hi):
        if lo >= rows:
            return
        if hi <= rows and (hi - lo == 1 or np.all(delta[lo:hi] == delta[lo])):
            load(lo, ctl)
            return
        mid = (lo + hi) // 2
        if rows <= mid:
            node(ctl, level + 1, lo, mid)
            return
        bit, a = bits[level], anc[level - 1]
        b.x(bit); b.init_and(ctl, bit, a); b.x(bit)
        node(a, level + 1, lo, mid)
        b.cx(ctl, a)
        node(a, level + 1, mid, hi)
        b.del_and(ctl, bit, a)

    half = 1 << (nbits - 1)
    b.x(bits[0]); node(bits[0], 1, 0, half)
    b.x(bits[0]); node(bits[0], 1, half, 2 * half)
    return b.finish_circuit()


def build_family(words, brot, J):
    return tuple(build_delta_circuit(d, brot) for d in batch_deltas(words, J))

In [ ]:
SELECTION = (words12.shape[0] - 1).bit_length()
print(f"selection {SELECTION} qubits, tree nodes <= {words12.shape[0]-1}\n")
print(f"{'brot':>4} {'J':>3} {'K':>3} {'peak Q':>7} {'spare':>6} {'T/circuit':>10} {'T (2 passes)':>13}")
rows = []
for brot, J in ((12, 3), (12, 8), (10, 3), (10, 8), (10, 10)):
    fam = build_family(requantise(words12, 12, brot), brot, J)
    costs = C.circuit_costs(fam)
    peak = max(c["num_qubits"] for c in costs)
    t2 = 2 * sum(c["max_magic"] for c in costs)
    rows.append(dict(brot=brot, J=J, K=len(fam), peak=peak, spare=128 - peak, T2=t2))
    print(f"{brot:4d} {J:3d} {len(fam):3d} {peak:7d} {128-peak:6d} {costs[0]['max_magic']:10d} {t2:13d}")

### Reading the sweep

Two rows do the work. `(brot 12, J 8)` needs 119 qubits and 10 circuits. `(brot 10, J 10)`
needs 123 and 8. Every row costs **3,974** Toffolis per circuit — one per tree node, minus
one node that early-break collapses — because the tree is over addresses, and nothing about
a word register changes how many nodes there are.

So the family cost is $2K \times 3{,}974$, and the whole game is $K$. Ten circuits → 79,480;
eight → 63,584. The 19.98% is bought with 4 qubits and two bits of precision. The qubits are in
surplus -- 5 of headroom. What the precision costs is settled in §6, and it is not in
surplus either: 242 mHa of one-body residual against a 20 mHa budget.

## 3. Does it load the table? Ask the competition's checker

`recover_words_from_circuits` simulates every address, applies the family forward and in
reverse, and requires: the words after circuit $k$ equal batch $k$ on every shot, the
selection register unchanged, the workspace back to $|0\rangle$, no uncorrected phase, the
reverse pass clearing the registers, and the padding beyond 75 zero. It raises
`InvalidCircuits` with the reason otherwise, so a pass is a statement about the circuits and
not about this notebook's bookkeeping.

In [ ]:
BROT, J = 10, 10
words = requantise(words12, 12, BROT)
family = build_family(words, BROT, J)

recovered, layout, toffoli = C.recover_words_from_circuits(
    family, num_vectors=words.shape[0], num_angles=words.shape[1],
    shots_per_address=2, seed=20261102)

print("register layout     : brot", layout.brot, "| J", layout.batch_size,
      "| selection", len(layout.selection))
print("words recovered     :", "EXACT" if np.array_equal(recovered, words) else "MISMATCH")
print("words compared      :", recovered.size)
print("toffoli, fwd+rev    :", int(round(toffoli)))
print("peak qubits         :", max(c["num_qubits"] for c in C.circuit_costs(family)))
print("reaction depth      :", max(c["reaction_depth"] for c in C.circuit_costs(family)))

## 4. Idea that was measured and died: 9-bit delta payloads

If every angle satisfied $\cos\theta > 0$ then $\theta \in (-\pi/2, \pi/2)$, and in turn
`bit 9 == bit 8` for a 10-bit word — so a delta bank could store 9 bits and rebuild the tenth
by copying, saving $3 \times 3 \times 1 = 9$ qubits, which is exactly the margin the banked
designs need. It is a clean argument and it does not survive the data.

In [ ]:
def sign_invariant_fraction(w, brot):
    return float((((w >> (brot - 1)) & 1) == ((w >> (brot - 2)) & 1)).mean())

for brot in (12, 10):
    w = requantise(words12, 12, brot)
    print(f"brot={brot:2d}: bit{brot-1} == bit{brot-2} on {sign_invariant_fraction(w, brot)*100:6.2f}% "
          f"of {w.size} words")

w10 = requantise(words12, 12, 10)
theta = w10 * (2 * np.pi / 2 ** BROT)
print(f"\ntheta/pi range: [{theta.min()/np.pi:.4f}, {theta.max()/np.pi:.4f}]  "
      "(the invariant's premise is |theta| < pi/2)")
print("words that break it:", int((~(((w10 >> 9) & 1) == ((w10 >> 8) & 1))).sum()))

**78.59% is a falsification, not a rounding hiccup.** This table is not a chain of Givens
rotations with positive cosine; its angles cover $\theta/\pi \in [0, 2)$, i.e. a full turn,
so the top bit is a genuine angle bit and 63,810 of 298,200 words need it stored. Any design
that budgets the 9 qubits back has to earn them somewhere else — the sweep above does that
instead, by buying circuit count with precision rather than payload with an invariant.

The lesson generalises: the sign argument was proven for the object it was stated about, and
then applied to a different object. A one-line empirical check would have caught it before it
reached a qubit budget.

## 5. What 10 bits actually costs

Precision is only free in `T`; it is not free in the answer. The step at `brot = 10` is
$2\pi/2^{10}$, and requantising from the table's 12 bits moves every angle by at most half of
that — measured below rather than assumed. What the step costs in *admissibility* is the other half of
the question, and §6 answers it.

In [ ]:
old = words12 * (2 * np.pi / 2 ** 12)
new = w10 * (2 * np.pi / 2 ** 10)
delta = (new - old + np.pi) % (2 * np.pi) - np.pi        # circular: a word is a turn count
print(f"max |dtheta|  = {np.abs(delta).max():.3e} rad")
print(f"mean |dtheta| = {np.abs(delta).mean():.3e} rad")
print(f"step          = {2*np.pi/2**BROT:.3e} rad")
print(f"fraction within half a step: {(np.abs(delta) <= np.pi/2**BROT).mean()*100:.2f}%")

## 6. The score, from the competition's own verifier

`verify.py` reads the factorization and the Givens tree from `submission.npz`, simulates the
circuits against all 3,976 vectors at 64 shots per address, decodes the words it recovers,
and evaluates them against the `FeMoCo76o113e.fcidump` shipped in the competition mount:

$$\text{score} = \lambda_{\text{eff}} \cdot T_{\text{total}} \cdot Q_{\text{total}}$$

with $\lambda_{\text{eff}} = \sqrt{E_{\text{gap}}(2\Lambda - E_{\text{gap}})}$, $T_{\text{total}}$
the ANGLES Toffolis plus the state-preparation and Givens costs, and $Q_{\text{total}}$ the
persistent state-preparation qubits plus the $J \cdot brot$ word registers plus the shared
workspace.

$\lambda_{\text{eff}}$ is a property of the words, so it is the one term precision moves;
$T_{\text{total}}$ and $Q_{\text{total}}$ are dominated by a state-preparation model that is
identical for every submission. And the score is not the only output: `error_1` and
`error_2` are gated at 20 mHa each, and a family that fails them scores `inf` however small
$T$ is. That gate is why the next cells score the whole in-cap frontier rather than the
cheapest family alone.

The scoring cells stop rather than substitute if the FCIDUMP is absent: `check_fcidump`
compares its size and SHA-256 and has no override argument, so there is no honest way to
score without it.


In [ ]:
out = pathlib.Path("submission"); out.mkdir(exist_ok=True)
for k, circuit in enumerate(family):
    circuit.to_file(C.kmb_path(out, k), format="kmb")
import shutil
shutil.copy(npz, out / "submission.npz")

import numpy as _np
with _np.load(npz, allow_pickle=False) as _z:
    _tree = _np.array(_z["tree_edges"], dtype=_np.int64)
digest = C.submission_sha256(
    [C.read_circuit(C.kmb_path(out, k), format="kmb") for k in range(len(family))],
    out / "submission.npz", _tree)
print("wrote", len(family), "circuits ->", out)
print("submission sha256:", digest)
print("fcidump present  :", (bundle / "verifiers" / "dfthcsos" / "FeMoCo76o113e.fcidump").is_file())

In [ ]:
import shutil, time
import numpy as np

FCIDUMP = bundle / "verifiers" / "dfthcsos" / "FeMoCo76o113e.fcidump"
assert FCIDUMP.is_file(), (
    f"the pinned FCIDUMP is not in this bundle ({FCIDUMP}); verify.py will not substitute one")

sys.path.insert(0, str(bundle / "verifiers"))
from dfthcsos.verify import evaluate_circuits, load_inputs, rejection_reason

# The factorization and the Givens tree are the same for every family below, so
# they are loaded once. verify() is exactly these three steps plus a raise, and
# evaluate_circuits is used directly so a rejected family still reports its
# error_1, error_2 and lambda_eff instead of only its rejection message.
FACTORIZATION = out / "submission.npz"
with np.load(FACTORIZATION, allow_pickle=False) as _z:
    TREE_EDGES = np.array(_z["tree_edges"], dtype=np.int64)
INPUTS = load_inputs(FACTORIZATION)


def score_family(label, directory):
    """Score one submission directory exactly as verify.py does, without raising."""
    started = time.time()
    result = evaluate_circuits(INPUTS, directory, TREE_EDGES, shots_per_address=64)
    metrics, resources = result["metrics"], result["resources"]
    reason = rejection_reason(metrics, resources)
    circuits = len(list(pathlib.Path(directory).glob("angle_k_*.kmb")))
    print(label)
    print(f"  status            : {'PASS' if reason is None else 'FAIL -- ' + reason}")
    print(f"  score             : {'inf' if reason else format(result['score'], '.6g')}")
    print(f"  lambda_eff        : {metrics['lambda_eff']:.6f}")
    print(f"  error_1 / error_2 : {metrics['error_1']*1e3:.4f} / {metrics['error_2']*1e3:.4f} mHa"
          f"   (limit 20)")
    print(f"  T_total           : {int(resources['toffoli']):,}"
          f"   (angles {int(resources['angles_toffoli']):,})")
    print(f"  Q_total           : {int(resources['qubits']):,}"
          f"   (angles {int(resources['angles_qubits'])}, cap 128)")
    print(f"  brot / J / K      : {result['brot']} / {result['batch_size']} / {circuits}")
    print(f"  verified shots    : {int(result['verified_shots']):,}")
    print(f"  submission sha256 : {result['submission_sha256']}")
    print(f"  elapsed           : {time.time() - started:.1f} s")
    return dict(brot=result["brot"], J=result["batch_size"], K=circuits,
                angles_qubits=int(resources["angles_qubits"]),
                angles_toffoli=int(resources["angles_toffoli"]),
                error_1=metrics["error_1"], error_2=metrics["error_2"],
                lambda_eff=metrics["lambda_eff"], score=result["score"],
                sha256=result["submission_sha256"], admitted=reason is None)

In [ ]:
# Every (brot, J) pair the cap admits, cheapest first: J is the largest batch that
# fits 128 qubits at that precision, since the number of circuits is ceil(75/J).
CANDIDATES = ((12, 8), (11, 9), (10, 10))

sweep = {}
for brot, J in CANDIDATES:
    directory = pathlib.Path(f"submission_b{brot}_J{J}")
    directory.mkdir(exist_ok=True)
    family_b = build_family(requantise(words12, 12, brot), brot, J)
    for k, circuit in enumerate(family_b):
        circuit.to_file(C.kmb_path(directory, k), format="kmb")
    shutil.copy(npz, directory / "submission.npz")
    sweep[(brot, J)] = score_family(f"brot {brot}, J {J}, {len(family_b)} circuits", directory)
    print()

In [ ]:
print(f"{'brot':>4} {'J':>2} {'K':>2} {'Q_ang':>5} {'T_ang':>7} {'err_1/mHa':>9}"
      f" {'err_2/mHa':>9} {'lambda':>7} {'status':>6} {'score':>12}")
for (brot, J), row in sweep.items():
    print(f"{brot:4d} {J:2d} {row['K']:2d} {row['angles_qubits']:5d} {row['angles_toffoli']:7d}"
          f" {row['error_1']*1e3:9.3f} {row['error_2']*1e3:9.3f} {row['lambda_eff']:7.3f}"
          f" {'PASS' if row['admitted'] else 'FAIL':>6}"
          f" {format(row['score'], '.6g') if row['admitted'] else 'inf':>12}")

control = score_family("control: the bundle's own sample submission",
                       bundle / "sample_submission" / "dfthcsos")

### What the sweep says

The 128-qubit cap admits three precisions, and one of them is admissible.

| `brot` | `J` | `K` | `Q` | `T` | `error_1` | `error_2` | score |
|---|---|---|---|---|---|---|---|
| 12 | 8 | 10 | 119 | 79,480 | 3.185 mHa | 18.640 mHa | **3.3556e9** |
| 11 | 9 | 9 | 122 | 71,532 | 221.29 mHa | 51.74 mHa | `inf` |
| 10 | 10 | 8 | 123 | 63,584 | 242.30 mHa | 59.38 mHa | `inf` |

The 12-bit family passes with `error_2` at 93% of its 20 mHa budget, so the table is not
merely rounded — it is near the edge of what the verifier accepts, and one bit off the end
is enough to leave it. Note what the one-body residual does: 3.19 mHa at twelve bits, 221
mHa at eleven. A 70x jump for one bit is not smooth degradation; pairing each submitted
one-body row with an exact eigenvector of `h1_prime` by overlap means that once the decoded
two-body block moves far enough, the assignment can switch to a different vector and the
residual jumps by the separation between eigenvalues rather than by the rounding error.
`error_2` moves the way rounding would (18.6 → 51.7 → 59.4). Either way the verdict is the
same at both coarser precisions, and it is `inf`.

So the two bits §2 spends on circuit count are not there to spend, and the notebook's
headline trade is falsified: `63,584` is a cheaper circuit family and an inadmissible
answer. The saving was real; the budget it was paid from was not.

That the 12-bit row and the control agree — same score, and the same `submission sha256`,
`276d464f…` — is the check that the pipeline is sound: two independent builders, one
written by the competition's `build_circuits.py` and one by `build_family` above, produce a
byte-identical submission from the same words.

The lesson has the same shape as §4's. A saving measured in `T` and `Q` is a hypothesis until
the accuracy gate is applied, and the gate was one function call away the whole time.


## 7. Where the next gain is

- **Precision is not the lever, and the cap was never the binding constraint.** Three
  precisions fit 128 qubits and only the one the bundle already ships is admissible (§6).
  At 12 bits the largest batch is `J = 8`, so `K = ceil(75/8) = 10` circuits and `T =
  79,480` are forced; there is no `(brot, J)` inside the cap below that, which is why this
  family is the floor for this table rather than a waypoint.
- **Refitting the words cannot rescue the coarser grid.** `scripts/refit_dfthcsos_words.py
  --oracle 12 11` builds the best submission each grid allows: it encodes the unquantised
  vectors on the tree, then rebuilds the one-body rows from the *decoded* two-body block —
  the order the shipped table uses. At 12 bits it reproduces the shipped family exactly
  (3.185 / 18.640 mHa), which says the construction is the real one; at 11 bits it lands at
  **25.0 mHa**, 5 mHa over the gate. No search beats an oracle, so the cheaper families are
  unreachable rather than unfitted. The reason is one level up: the unquantised vector set
  is itself at 16.13 mHa, leaving 3.87 mHa of headroom for a whole bit of grid to eat.
- **One ancilla per tree is impossible, not merely expensive.** Sharing a single ancilla
  across the 11 levels was tried and kickmix refuses to build it: the parent's control has to
  survive its whole left subtree, so a child writing that qubit makes the parent's control and
  the child's target the same qubit (`CCX q122 q9 q122`, reported 7,943 times). The 11
  ancillas are not overhead — they are what makes the tree a tree.
- **Early-break has nothing to collapse.** All 3,976 delta rows are distinct in every one of
  the 8 circuits — 31,808 loads, 31,808 distinct — so a tree built over the delta tables
  finds no repeated subtree to share. The 3,974 Toffolis per circuit is the padding-trimmed
  tree (3,976 rows in 4,096 leaves), not a data-dependent saving.
  The break now compares whole rows rather than cells, which is strictly stronger and is what
  the ladder prototype uses; `--break-audit` shows it finds **0** collapsible ranges in each
  of the circuits, so the count above is unchanged.
- **A sublinear-$T$ read was built, and on this table it loses.** The ladder read computes
  $\sqrt{A}$ offset controls once and keeps them alive, then loads each block by `ccx`.
  Addressing falls 21.7x (3,974 $\to$ 183 Toffolis), but a load with two controls is a
  Toffoli, so at this table's 40.0 set bits per address the read costs 40.3x the tree
  (160,064 against 3,974 at `J = 10`; 64,115 at `J = 4`). Given the same early-break rule
  this tree uses, it wins only below 0.87 set bits per address -- 0.84 without the break,
  0.95 if the tree cannot collapse at all -- so the bracket is 42x to 46x below what the
  table carries, whichever end is right.
- **What decides those numbers is empty addresses, not repeated words.** Scattering every
  set bit to its own bit position, so no two rows are equal, moves nothing (tree 1,405 at
  0.10 and 3,744 at 1.00 either way): a sparse table has all-zero blocks whatever its
  words look like. Filling one bit into *every* address is the one regime where the
  ladder was cheaper by 8% (3,638 against 3,974); with the tree breaking on row equality that
  tree is 3,453 and the ladder is 5% dearer -- and 40 bits on every address, which is
  this table, is 40x past it. Any sparser table that would be cheap has empty addresses by
  construction, so the two conditions cannot be met at once.
- **On the shipped `J = 10` family that window closes.** 100 payload qubits leave only
  `low = 2` inside the cap, so the address tree costs 996 Toffolis instead of 183: the
  crossing drops to 0.56, and the one-bit-per-address table that was 8% cheaper at
  `J = 4, low = 5` is 26% dearer (4,765 against 3,770). The bracket for the family that
  actually ships is `[0.56, 0.75]`. Structured payloads are also closer than the
  uncollapsed read suggested -- 280 against 131 with eight runs, 20 against 14 with one,
  now that the ladder's own offset tree is built only where a load addresses it -- but
  they still lose, so the ladder loses however this table is arranged.
- **`low = 2` was not the split to choose.** Costing every split with its qubit count
  shows the cap admits only `low <= 2` at `J = 10`, and the best of those is `low = 0` --
  the tree again (3,486 Toffolis at 0.75 bits per address, 3,770 at one bit on every
  address, against 4,765 at `low = 2`). No in-cap split wins on the shipped family; the
  ladder's advantage first appears at `low = 3`, which needs 134 qubits. So the 0.56
  crossing describes a split that is not the best one available.
- **Three of the four layouts never pick the ladder.** `--low-matrix` walks all eight
  densities on all four payload layouts. At `J = 10` the best in-cap split is `low = 0` --
  the tree -- for per-address (386-3,770 Toffolis), blocked (89-122) and contiguous (15) at
  every density; only the random layout ever chooses `low = 2` (0.79-0.96x at 0.10-0.50).
  The same grid at `J = 4`, where `low = 5` does fit, moves the random window out to 0.75
  bits per address (0.41-0.89x) and changes nothing for the structured tables: their trees
  are already 14-131 Toffolis, less than the ladder's 183-Toffoli addressing. Structure is
  not sparsity with order — it is the case the tree is near-optimal on.
- **A flat comparator table is worse on both axes.** At `low = 12`, where the block count
  exceeds the rows, the read degenerates to one comparator per address: 6,980 and 7,548
  Toffolis at 7,094 and 7,662 qubits, against 3,691 and 3,770 at 123 for the tree. That is
  58x the qubits for no fewer Toffolis, so no table here is worth asking the cap for.
- **The second pass cannot be replaced by measurement-based uncomputation.** This is not a
  gate-set limit. The checker applies the family in reverse and asserts after *every* step that
  the word registers hold the previous batch (`reverse ANGLE_k did not restore the ANGLE_{k-1}
  word registers`), and that the final step clears them. A measurement-based clear produces
  zeros, so it fails at the first `k > 0` however the qubits are measured; and the reported
  Toffoli count is the forward and reverse passes together divided by shots, so the factor of 2
  is structural rather than a bookkeeping choice. The pinned gate set would block it anyway:
  there is no Z-basis read, and `R` randomises the tracked phase.
  Nor can the two applications share an address tree. Between the forward and reverse use of
  one circuit nothing differs — the checker re-executes the same bytes — so there is no second
  payload to fuse; the `T` charge is exactly `2 * sum(circuit Toffolis)`, and a family one
  circuit short is rejected against `ceil(75/J)` rather than completed by the sweep.

The digest in the last line is the value the verifier seeds its simulator from, so it is a
checkpoint: re-running the builder must reproduce it byte for byte, and every score in §6 is
a function of it. `fcidump present: True` is what makes the scoring cells run — the same
line reads `False` in a checkout without the competition mount, which is why the guard in §6
fails loudly rather than falling back to a stand-in.
